In [ ]:
# MySQL -> SQL, indexing 가능 : 방대한 데이터 , 특정 데이터 찾아오려고 할때 시간 효율성 극대화 목적
#MongoDB -> Nosql,indexing기능 

#인덱스 생성 : 
>단일 인덱스 : 인덱스 조건 1개
>복합 인덱스 : 인덱스 조건 복합적인 요소로 구성
>>다중키 안덱스: 배열 안에 속해잇는 값들을 인덱스의 조건으로 설정
>>덱스트 인덱스 : 문자열을 활용항 인덱스 설정

#인덱스 삭제 : 
#인덱스  리시트 조회:

In [47]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")
db = client.sample_mflix
movies = db.movies



In [48]:
result = movies.create_index("title")
print(result) # 결과 괎: 어떤 컬럼을 인덱스의 대사응, 1=> 정렬방향 , 오름차순

title_1


In [49]:
result = movies.create_index([("title",-1)])
print(result)

title_-1


In [50]:
indexes = movies.index_information()
print(indexes)

{'_id_': {'v': 2, 'key': [('_id', 1)]}, 'title_1': {'v': 2, 'key': [('title', 1)]}, 'title_-1': {'v': 2, 'key': [('title', -1)]}}


result = movies.create_index([("title",1),{"year",-1}])

In [51]:
for movie in movies.aggregate([
    {"$limit": 1}
]) : 
    print(movie["cast"])

['Charles Kayser', 'John Ott']


In [52]:
#배열안에 입력된 값들 주 ㅇ1개라도 있으면 보다 빠르게 값을 조회, 수집

result = movies.create_index("cast")
print(result)

cast_1


In [53]:
#문자열로 구성된 컨텐츠 가운데, 해당 단어들이 속해져 있으면 값을 조회, 수집
result = movies.create_index([("plot", "text")])
print(result)

plot_text


In [54]:
movies.drop_indexes()

In [55]:
indexes = movies.index_information()
print(indexes)

{'_id_': {'v': 2, 'key': [('_id', 1)]}}


In [ ]:
#인덱스 세팅시 가중차 ㄱ밧을 부여 -> 특정인덱스의 경우 더 높은 점수를 부여
#MongoDb > 디비> 컬렉션> 데이터 저장 (상품명, 상세페이지 문안,상품가격, 할인가겨)
#젤리 키워드 > 결과값 출력!! 
#조회 목적 : 사용자, 우리 사이트 방문 > 인기 키워드 검색 시, 상위노출 상품패턴 분석


In [56]:
movies.create_index([("title", "text"),("plot", "text")], weights = {"title":5, "plot":1})

'title_text_plot_text'

In [57]:
indexes = movies.index_information()
print(indexes)

{'_id_': {'v': 2, 'key': [('_id', 1)]}, 'title_text_plot_text': {'v': 2, 'key': [('_fts', 'text'), ('_ftsx', 1)], 'weights': SON([('plot', 1), ('title', 5)]), 'default_language': 'english', 'language_override': 'language', 'textIndexVersion': 3}}


In [60]:
cursor = movies.find(
    {"$text": {"$search":"thriller"}},
    {"score:": {"$meta":"textScore"}}).sort([("score", {"$meta":"textScore"})]).limit(5)

for doc in cursor :
    print(doc)



{'_id': ObjectId('573a1398f29313caabce9497'), 'plot': 'A night at the movies turns into a nightmare when Michael and his date are attacked by a hoard of bloody-thirsty zombies - only "Thriller" can save them now.', 'genres': ['Short', 'Horror', 'Music'], 'runtime': 13, 'rated': 'PG', 'cast': ['Michael Jackson', 'Ola Ray', 'Vincent Price', 'Hanala Sagal'], 'poster': 'https://m.media-amazon.com/images/M/MV5BODhhZjJlYTktZDQ2MS00Yzk4LWFlOTQtYTgyOGE1ZGE5YWEyL2ltYWdlXkEyXkFqcGdeQXVyMzA5MjgyMjI@._V1_SY1000_SX677_AL_.jpg', 'title': 'Michael Jackson: Thriller', 'fullplot': 'Michael Jackson and his date are watching a movie. They leave, and take a shortcut through the graveyard on the way home. Michael turns into a werepanther-type creature, and then later a zombie, as he gets down and funky in a tremendous dance scene to the tune of his song "Thriller."', 'languages': ['English'], 'released': datetime.datetime(1983, 12, 2, 0, 0), 'directors': ['John Landis'], 'writers': ['John Landis', 'Michael

In [ ]:
#1) 해당 키워드가 많이 등장할 수록 점수 획득
#2) 해당 키워드가 전체 데이터 상 희소할 수록 검색 발견시, 점수 높게 측정
#3) 방금 설정한 필드 가중치 값 부여(상수값, 비율)
#-> 1,2,3 종합=> 몽고디비만의 인덱스 설정 엔진활용 => 최종 스코어 값!